# fz-MCNP Plugin Example Usage

This notebook demonstrates how to use the fz-MCNP plugin for parametric MCNP calculations.

The examples cover:
1. Installing dependencies (fz framework and fz-MCNP plugin)
2. Downloading example input files
3. Parsing input files to identify variables
4. Compiling input files with specific parameter values
5. Running calculations with MCNP
6. Performing parametric studies

**Note:** running calculations (examples 3 and 4) requires MCNP: `mcnp6` on `PATH` (or `MCNP_CMD` / `MCNP_PATH`) and its cross-section data (`DATAPATH`).

## Setup: Install Dependencies

First, install the Funz framework and the fz-MCNP plugin.

In [ ]:
# Install fz framework from GitHub
%pip install -q git+https://github.com/Funz/fz.git

# Install fz-MCNP plugin
import fz
fz.install("MCNP")

print("✓ Dependencies installed successfully!")

## Setup: Download Example Files

Download the example input files from the GitHub repository.

In [ ]:
import os
import urllib.request

# Base URL for raw GitHub content
base_url = "https://raw.githubusercontent.com/Funz/fz-MCNP/main/examples"

# Download example files
files_to_download = [
    ("godiva.inp", f"{base_url}/godiva.inp"),
]

print("Downloading example files...")
for local_path, url in files_to_download:
    try:
        urllib.request.urlretrieve(url, local_path)
        print(f"  ✓ Downloaded: {local_path}")
    except Exception as e:
        print(f"  ✗ Failed to download {local_path}: {e}")

print("\n✓ Example files ready!")

## Setup: Configure MCNP

The calculator script (`.fz/calculators/MCNP.sh`) finds MCNP through these environment variables:

| Variable | Role |
|---|---|
| `MCNP_CMD` | MCNP executable (default: `mcnp6` on `PATH`) |
| `MCNP_PATH` | installation directory with `MCNP_CODE/bin/mcnp6` and `MCNP_DATA/` (legacy layout) |
| `DATAPATH` | directory holding `xsdir` (default: `$MCNP_PATH/MCNP_DATA`) |
| `MCNP_TASKS` | number of threads |

Set them only if needed (values below are examples).

In [ ]:
import os

# Uncomment and adjust if mcnp6 is not on PATH:
# os.environ['MCNP_CMD'] = '/opt/MCNP6/bin/mcnp6'
# os.environ['DATAPATH'] = '/opt/MCNP6/MCNP_DATA'

for var in ('MCNP_CMD', 'MCNP_PATH', 'DATAPATH', 'MCNP_TASKS'):
    print(f"{var} = {os.environ.get(var, '<not set>')}")

## Import Required Libraries

In [ ]:
import fz
import os
import tempfile

print(f"fz version: {fz.__version__ if hasattr(fz, '__version__') else 'unknown'}")

## Example 1: Parse Input File for Variables

Use `fz.fzi()` to identify variables in an MCNP input file.

In [ ]:
# Parse the input file to find variables
variables = fz.fzi(
    "godiva.inp",
    "MCNP"
)

print(f"Variables found in godiva.inp: {list(variables.keys())}")

## Example 2: Compile Input Files

Use `fz.fzc()` to compile input files with specific parameter values. This substitutes variables with actual values.

In [ ]:
# Create temporary directory for compiled files
with tempfile.TemporaryDirectory() as tmpdir:
    # Compile input with specific value for radius
    fz.fzc(
        "godiva.inp",
        {"r": 8.741},  # Critical radius
        "MCNP",
        output_dir=tmpdir
    )
    
    # Show compiled file
    # Find the compiled file (it may be in a subdirectory)
    for root, dirs, files in os.walk(tmpdir):
        for f in files:
            if f == "godiva.inp":
                compiled_file = os.path.join(root, f)
                print(f"Compiled file content (r=8.741):")
                print("=" * 70)
                with open(compiled_file, 'r') as file:
                    print(file.read())

## Example 3: Run a Basic MCNP Calculation

Run a simple MCNP calculation with a single parameter value.

**Note:** This requires MCNP to be installed and properly configured.

In [ ]:
# Define parameter value
input_variables = {
    "r": [8.7407]  # Godiva critical radius (cm)
}

try:
    # Run MCNP calculation
    results = fz.fzr(
        input_path="godiva.inp",
        input_variables=input_variables,
        model="MCNP",
        calculators="localhost_MCNP",
        results_dir="results/basic_study"
    )
    
    # Display results
    print("\nResults:")
    print("=" * 70)
    print(results[['r', 'status', 'mcnp_status', 'mean_keff', 'sigma_keff']])
    
except Exception as e:
    print(f"\nError running calculation: {e}")
    print("Note: Make sure MCNP is installed and MCNP_PATH is set correctly.")

## Example 4: Parametric Study of Godiva Criticality

Demonstrate a parametric study with multiple sphere radii to study criticality.

In [ ]:
import numpy as np

# Define multiple radius values around the critical radius
input_variables = {
    "r": [8.5, 8.7407, 9.0]  # around the critical radius
}

print(f"Running parametric study...")
print(f"  r values: {input_variables['r']}")

try:
    # Run parametric calculation
    results = fz.fzr(
        input_path="godiva.inp",
        input_variables=input_variables,
        model="MCNP",
        calculators="localhost_MCNP",
        results_dir="results/parametric_study"
    )
    
    # Display results
    print("\nResults:")
    print("=" * 70)
    print(results)
    
    # Analyze criticality
    print("\nCriticality Analysis:")
    print("=" * 70)
    for idx, row in results.iterrows():
        r = row['r']
        keff = row.get('mean_keff', None)
        sigma = row.get('sigma_keff', None)
        if keff is not None and keff != '':
            try:
                keff_val = float(keff)
                sigma_val = float(sigma)
                status = "supercritical" if keff_val > 1.0 else "subcritical" if keff_val < 1.0 else "critical"
                print(f"  r={r:.3f} cm: k_eff = {keff_val:.4f} ± {sigma_val:.4f} ({status})")
            except:
                print(f"  r={r:.3f} cm: Calculation incomplete")
    
except Exception as e:
    print(f"\nError running parametric study: {e}")
    print("Note: Make sure MCNP is installed and MCNP_PATH is set correctly.")

## Example 5: Visualization

Plot keff with its standard deviation from the results of Example 4.

In [ ]:
import matplotlib.pyplot as plt

# Plot the results of Example 4 (cases without keff are skipped)
ok = results[results['mcnp_status'] == 'NORMAL END']
if ok.empty:
    print('No successful MCNP run to plot.')
else:
    plt.figure(figsize=(8, 5))
    plt.errorbar(ok['r'].astype(float), ok['mean_keff'].astype(float),
                 yerr=ok['sigma_keff'].astype(float), fmt='o-', capsize=5)
    plt.axhline(y=1.0, color='r', linestyle='--', label='keff = 1')
    plt.xlabel('Sphere radius (cm)')
    plt.ylabel('keff')
    plt.title('Godiva: keff vs radius')
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.show()